In [8]:
import yfinance as yf
import pandas as pd

# 1. Fetch the current S&P 500 ticker list from Wikipedia
raw_path = "data/raw/sp500list.csv"
tables = pd.read_csv(raw_path)
sp500_table = tables["symbol"]

# Yahoo Finance uses hyphens instead of dots for share classes (e.g., BRK-B instead of BRK.B)
tickers = sp500_table.tolist()

# 2. Download historical data for all tickers
# Adjust start, end, or period (e.g., period="5y", period="max") as needed
data = yf.download(
    tickers=tickers,
    start="2020-01-01",
    end="2026-01-01",
    interval="1d",       # Options: 1d, 1wk, 1mo
    group_by="ticker",   # Organizes columns by ticker symbol first
    auto_adjust=True,    # Adjusts Open/High/Low/Close for splits and dividends
    threads=True         # Uses multithreading for faster bulk downloads
)

# 3. Save the dataset to a CSV file
data.to_csv("sp500_historical_data.csv")
print("Download complete!")


[                       1%                       ]  4 of 502 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BF.B"}}}
$BF.B: possibly delisted; no timezone found
[**                     4%                       ]  22 of 502 completed$ABC: possibly delisted; no timezone found
[***                    7%                       ]  36 of 502 completed$HES: possibly delisted; no timezone found
[***                    7%                       ]  37 of 502 completed$BRK.B: possibly delisted; no timezone found
[****                   8%                       ]  41 of 502 completed$QRVO: Data doesn't exist for startDate = 1577854800, endDate = 1767243600
[*****                 10%                       ]  50 of 502 completed$JNPR: possibly delisted; no timezone found
[*****                 10%                       ]  52 of 502 completed$ABMD: possibly delisted; no timezone found
[******                13%             

Download complete!


In [9]:
import pandas as pd
import numpy as np

# 1. Load the raw multi-index CSV
raw_path = "data/raw/sp500_historical_data.csv"
raw_df = pd.read_csv(raw_path, header=[0, 1], index_col=0, parse_dates=True)

# 2. Extract only the 'Close' price slice across all 502 tickers
close_prices = raw_df.xs("Close", axis=1, level=1)

# 3. Clean & filter data: remove stocks with missing data or short histories
# (Drops 43 tickers such as BF.B, BRK.B, HES, ABMD that have missing/delisted dates)
clean_prices = close_prices.dropna(axis=1, how="any")

# 4. Standardize order and validate data integrity
clean_prices = clean_prices.sort_index(ascending=True)          # Chronological dates
clean_prices = clean_prices.reindex(sorted(clean_prices.columns), axis=1)  # Alphabetical tickers
clean_prices = clean_prices.loc[:, (clean_prices > 0).all()]    # Ensure strictly positive prices for log()

# 5. Save to a clean, single-header CSV for the next phase
clean_path = "data/processed/sp500_clean_close.csv"
clean_prices.to_csv(clean_path)

print(f"Saved clean dataset to '{clean_path}'")
print(f"Dimensions: {clean_prices.shape[0]} trading days x {clean_prices.shape[1]} tickers")
print(f"Date range: {clean_prices.index.min().date()} to {clean_prices.index.max().date()}")

Saved clean dataset to 'data/processed/sp500_clean_close.csv'
Dimensions: 1508 trading days x 459 tickers
Date range: 2020-01-02 to 2025-12-31


In [12]:
import pandas as pd
import numpy as np

# 1. Load your cleaned closing prices
prices = pd.read_csv("data/processed/sp500_clean_close.csv", index_col=0, parse_dates=True)

# 2. Convert to natural log prices
log_prices = np.log(prices)

# 3. Chronological 70% In-Sample / 30% Out-of-Sample split
split_idx = int(len(log_prices) * 0.70)
in_sample = log_prices.iloc[:split_idx]
out_of_sample = log_prices.iloc[split_idx:]

in_sample.to_csv("data/processed/sp500_in_sample.csv")
out_of_sample.to_csv("data/processed/sp500_out_of_sample.csv")

print(f"In-sample period:     {in_sample.index.min().date()} to {in_sample.index.max().date()} ({len(in_sample)} days)")
print(f"Out-of-sample period: {out_of_sample.index.min().date()} to {out_of_sample.index.max().date()} ({len(out_of_sample)} days)")

In-sample period:     2020-01-02 to 2024-03-12 (1055 days)
Out-of-sample period: 2024-03-13 to 2025-12-31 (453 days)
